In [ ]:
# 第一个单元格：导入必要的库
import os
import numpy as np
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
import matplotlib.pyplot as plt
from tqdm import tqdm
import pickle
import pandas as pd
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import seaborn as sns

# 导入自定义的数据加载模块
from data_loader import load_and_prepare_data, analyze_data_distribution

# 设置随机种子以确保结果可重复
torch.manual_seed(666)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(666)
np.random.seed(666)

# 设置设备
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"使用设备: {device}")

In [ ]:
# 第二个单元格：定义模型架构
class DenseModel(nn.Module):
    """模仿原始Keras模型的PyTorch实现"""
    def __init__(self, input_dim, hidden_dim=4096, num_classes=102, dropout_rate=0.5):
        super(DenseModel, self).__init__()
        self.layer1 = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer2 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer3 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.layer4 = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate)
        )
        self.output_layer = nn.Linear(hidden_dim, num_classes)
        
        # 初始化权重
        self._initialize_weights()
    
    def _initialize_weights(self):
        for m in self.modules():
            if isinstance(m, nn.Linear):
                nn.init.kaiming_normal_(m.weight, mode='fan_in', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)
        x = self.layer4(x)
        x = self.output_layer(x)
        return torch.softmax(x, dim=1)

In [ ]:
# 第三个单元格：数据标准化检查函数
def check_standardization(data, threshold=0.1, sample_size=1000):
    """
    检查数据是否已标准化

    参数:
        data: numpy数组或PyTorch张量，形状为 [n_samples, n_features]
        threshold: 均值和标准差允许的偏差阈值
        sample_size: 检查的样本数量，如果数据很大，我们只检查一部分

    返回:
        bool: 数据是否已标准化
    """
    # 如果是PyTorch张量，转换为numpy数组
    if isinstance(data, torch.Tensor):
        data = data.cpu().numpy()
    
    # 如果数据量大，随机抽样
    if len(data) > sample_size:
        indices = np.random.choice(len(data), sample_size, replace=False)
        data = data[indices]
    
    # 计算每个特征的均值和标准差
    means = np.mean(data, axis=0)
    stds = np.std(data, axis=0)
    
    # 检查均值是否接近0，标准差是否接近1
    mean_close_to_zero = np.all(np.abs(means) < threshold)
    std_close_to_one = np.all(np.abs(stds - 1.0) < threshold)
    
    # 打印统计信息
    print(f"特征均值范围: [{means.min():.4f}, {means.max():.4f}], 平均={np.mean(means):.4f}")
    print(f"特征标准差范围: [{stds.min():.4f}, {stds.max():.4f}], 平均={np.mean(stds):.4f}")
    
    return mean_close_to_zero and std_close_to_one

In [ ]:
# 第四个单元格：数据加载和标准化检查，添加标签分析
# 设置数据路径
base_dir = "/home/jovyan/gpu_space/workspace_jiayi/KAN training/brain_voxel_data/processed_data"  # 请根据实际路径修改
format = 'mat'  # 保持与原始代码一致

# 加载数据
create_new_scaler = False  # 设置为False使用现有scaler，设置为True创建新scaler
train_loader, val_loader, test_loader, feature_dim, num_classes, scaler = load_and_prepare_data(
    base_dir=base_dir, 
    create_new_scaler=create_new_scaler,
    format=format,
    batch_size=128,  # 与原始代码相同的batch_size
    shuffle=True,
    seed=666
)

print(f"特征维度: {feature_dim}")
print(f"数据加载器返回的类别数量: {num_classes}")

# 检查所有数据加载器中的最大标签值
max_label = -1
for loader_name, loader in [("训练集", train_loader), ("验证集", val_loader), ("测试集", test_loader)]:
    for _, labels in loader:
        batch_max = labels.max().item()
        if batch_max > max_label:
            max_label = batch_max
        break  # 只检查第一个批次
    print(f"{loader_name}第一批次的最大标签值: {batch_max}")

print(f"\n数据集中的最大标签值: {max_label}")
print(f"所需的模型输出类别数: {max_label + 1} (标签范围 0-{max_label})")

# 检查数据是否已标准化
print("\n检查训练集数据标准化:")
train_data, train_labels = next(iter(train_loader))
is_train_standardized = check_standardization(train_data, threshold=0.25)
print(f"训练集数据是否已标准化 (阈值0.25): {is_train_standardized}")

# 收集更多标签样本以获得更全面的统计
print("\n分析标签分布...")
all_labels = []
for i, (_, labels) in enumerate(train_loader):
    all_labels.append(labels)
    if i >= 10:  # 只收集前10个批次
        break
all_labels = torch.cat(all_labels)

label_range = (all_labels.min().item(), all_labels.max().item())
print(f"标签范围 (min-max): {label_range}")

# 计算标签分布
label_counts = torch.bincount(all_labels.flatten())
unique_labels = (label_counts > 0).sum().item()
print(f"不同标签的数量: {unique_labels}")

# 显示标签分布
plt.figure(figsize=(15, 5))
plt.bar(range(len(label_counts)), label_counts.numpy())
plt.title("标签分布")
plt.xlabel("标签索引")
plt.ylabel("样本数量")
plt.show()

# 分析数据分布（可选）
# 如果你想了解更多关于数据分布的信息
print("\n运行全面的数据分析...")
from data_loader import analyze_data_distribution
analysis_results = analyze_data_distribution(train_loader, val_loader, test_loader)

In [ ]:
# 第五个单元格：模型定义、训练函数和评估函数
# 创建模型 - 确保类别数量充分
model = DenseModel(
    input_dim=feature_dim,
    hidden_dim=4096,  # 与原始模型相同
    num_classes=102,  # 确保类别数量足够 (0-101)
    dropout_rate=0.5  # 与原始模型相同
).to(device)

print(model)
print(f"输出层大小: {model.output_layer.out_features} 类别，覆盖标签范围 0-{model.output_layer.out_features-1}")

# 设置优化器和损失函数
# 使用与原始模型相同的学习率
optimizer = optim.Adam(model.parameters(), lr=0.00001, weight_decay=0.00001)  # L2正则化
criterion = nn.CrossEntropyLoss()

# 训练函数
def train_one_epoch(model, dataloader, criterion, optimizer, device):
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 使用tqdm显示进度条
    pbar = tqdm(dataloader, desc="Training")
    
    for batch_idx, (inputs, targets) in enumerate(pbar):
        # 首批次时检查标签范围
        if batch_idx == 0:
            max_label = torch.max(targets).item()
            min_label = torch.min(targets).item()
            print(f"[信息] 第一批次标签范围: {min_label} 到 {max_label}")
            if max_label >= model.output_layer.out_features:
                print(f"警告: 发现标签值 {max_label} 超出模型输出类别数 {model.output_layer.out_features}")
                print(f"标签分布: {torch.bincount(targets.flatten())}")
                raise ValueError(f"标签索引超出模型输出类别范围，请增加类别数至少到 {max_label+1}")
            
        inputs, targets = inputs.to(device), targets.to(device)
        
        # 梯度清零
        optimizer.zero_grad()
        
        # 前向传播
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # 反向传播和优化
        loss.backward()
        optimizer.step()
        
        # 统计
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # 更新进度条
        pbar.set_postfix({
            'loss': running_loss / (batch_idx + 1),
            'acc': 100. * correct / total
        })
    
    return running_loss / len(dataloader), 100. * correct / total

# 评估函数
def evaluate(model, dataloader, criterion, device):
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    # 禁用梯度计算
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Evaluation"):
            # 检查最大标签
            max_label = torch.max(targets).item()
            if max_label >= model.output_layer.out_features:
                print(f"警告: 评估时发现标签值 {max_label} 超出模型输出类别数 {model.output_layer.out_features}")
                continue
                
            inputs, targets = inputs.to(device), targets.to(device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 统计
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    return running_loss / len(dataloader), 100. * correct / total

In [ ]:
# 第六个单元格：训练模型（添加macro-F1评估）
from sklearn.metrics import f1_score

# 增强评估函数以返回预测和实际标签，用于计算F1分数
def evaluate_with_predictions(model, dataloader, criterion, device):
    """评估模型并返回预测和标签，便于计算F1分数"""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    all_preds = []
    all_targets = []
    
    # 禁用梯度计算
    with torch.no_grad():
        for inputs, targets in tqdm(dataloader, desc="Evaluation"):
            # 检查最大标签
            max_label = torch.max(targets).item()
            if max_label >= model.output_layer.out_features:
                print(f"警告: 评估时发现标签值 {max_label} 超出模型输出类别数 {model.output_layer.out_features}")
                continue
            
            inputs, targets = inputs.to(device), targets.to(device)
            
            # 前向传播
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            # 获取预测
            _, predicted = outputs.max(1)
            
            # 收集预测和目标
            all_preds.append(predicted.cpu().numpy())
            all_targets.append(targets.cpu().numpy())
            
            # 统计
            running_loss += loss.item()
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
    
    # 合并所有批次的预测和目标
    all_preds = np.concatenate(all_preds)
    all_targets = np.concatenate(all_targets)
    
    return running_loss / len(dataloader), 100. * correct / total, all_preds, all_targets

# 训练参数
num_epochs = 25  # 与原始模型相同
best_val_acc = 0.0
best_val_f1 = 0.0
train_losses = []
train_accs = []
train_f1s = []
val_losses = []
val_accs = []
val_f1s = []

# 训练循环
for epoch in range(num_epochs):
    print(f"\nEpoch {epoch+1}/{num_epochs}")
    
    # 训练阶段
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer, device)
    train_losses.append(train_loss)
    train_accs.append(train_acc)
    
    # 训练集的F1评估（采样一部分计算，以节省时间）
    _, _, train_preds, train_targets = evaluate_with_predictions(model, 
                                                                DataLoader(train_loader.dataset, 
                                                                          batch_size=train_loader.batch_size, 
                                                                          sampler=torch.utils.data.RandomSampler(
                                                                              train_loader.dataset, 
                                                                              num_samples=min(10000, len(train_loader.dataset)))), 
                                                                criterion, device)
    train_f1 = f1_score(train_targets, train_preds, average='macro')
    train_f1s.append(train_f1)
    
    # 验证阶段（完整评估）
    val_loss, val_acc, val_preds, val_targets = evaluate_with_predictions(model, val_loader, criterion, device)
    val_losses.append(val_loss)
    val_accs.append(val_acc)
    
    # 计算验证集的宏观F1
    val_f1 = f1_score(val_targets, val_preds, average='macro')
    val_f1s.append(val_f1)
    
    print(f"Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%, Train Macro-F1: {train_f1:.4f}")
    print(f"Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%, Val Macro-F1: {val_f1:.4f}")
    
    # 保存最佳模型 - 同时考虑准确率和F1分数
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        print(f"Saving model with best validation accuracy: {val_acc:.2f}%")
        torch.save(model.state_dict(), 'dense_4x4096_model_best_acc.pth')
    
    if val_f1 > best_val_f1:
        best_val_f1 = val_f1
        print(f"Saving model with best validation Macro-F1: {val_f1:.4f}")
        torch.save(model.state_dict(), 'dense_4x4096_model_best_f1.pth')

# 绘制训练过程 - 包括F1分数
plt.figure(figsize=(18, 5))

# 损失曲线
plt.subplot(1, 3, 1)
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Val Loss')
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('Training and Validation Loss')
plt.legend()

# 准确率曲线
plt.subplot(1, 3, 2)
plt.plot(train_accs, label='Train Acc')
plt.plot(val_accs, label='Val Acc')
plt.xlabel('Epoch')
plt.ylabel('Accuracy (%)')
plt.title('Training and Validation Accuracy')
plt.legend()

# F1分数曲线
plt.subplot(1, 3, 3)
plt.plot(train_f1s, label='Train Macro-F1')
plt.plot(val_f1s, label='Val Macro-F1')
plt.xlabel('Epoch')
plt.ylabel('F1 Score')
plt.title('Training and Validation Macro-F1 Score')
plt.legend()

plt.tight_layout()
plt.show()

# 训练结束后，打印最佳结果
print(f"最佳验证准确率: {best_val_acc:.2f}%")
print(f"最佳验证Macro-F1: {best_val_f1:.4f}")

# 对类别分布进行分析
plt.figure(figsize=(12, 6))
plt.hist(val_targets, bins=num_classes, alpha=0.5, label='True Class Distribution')
plt.hist(val_preds, bins=num_classes, alpha=0.5, label='Predicted Class Distribution')
plt.xlabel('Class Index')
plt.ylabel('Count')
plt.title('True vs Predicted Class Distribution')
plt.legend()
plt.xticks(np.arange(0, num_classes, step=5))
plt.grid(True, alpha=0.3)
plt.show()

# 计算每个类别的F1分数
class_f1 = f1_score(val_targets, val_preds, average=None)

# 找出表现最好和最差的类别
top_classes = np.argsort(class_f1)[-10:]  # 最好的10个类别
bottom_classes = np.argsort(class_f1)[:10]  # 最差的10个类别

# 可视化
plt.figure(figsize=(15, 10))

# 所有类别的F1分数
plt.subplot(2, 1, 1)
plt.bar(range(len(class_f1)), class_f1)
plt.xlabel('Class Index')
plt.ylabel('F1 Score')
plt.title('F1 Score for Each Class')
plt.grid(True, alpha=0.3)

# 最好和最差的类别对比
plt.subplot(2, 1, 2)
combined = np.concatenate([bottom_classes, top_classes])
combined_f1 = class_f1[combined]
colors = ['r'] * 10 + ['g'] * 10  # 红色表示最差，绿色表示最好

plt.bar(range(len(combined)), combined_f1, color=colors)
plt.xticks(range(len(combined)), combined, rotation=45)
plt.xlabel('Class Index')
plt.ylabel('F1 Score')
plt.title('Best and Worst Performing Classes')
plt.axhline(y=np.mean(class_f1), color='b', linestyle='--', label=f'Average F1: {np.mean(class_f1):.4f}')
plt.legend()
plt.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 第七个单元格：加载最佳模型并在测试集上评估
# 加载最佳模型
model.load_state_dict(torch.load('dense_4x4096_model_pytorch.pth'))

# 在测试集上评估
test_loss, test_acc = evaluate(model, test_loader, criterion, device)
print(f"Test Loss: {test_loss:.4f}, Test Acc: {test_acc:.2f}%")

# 获取详细的分类报告
all_preds = []
all_targets = []
model.eval()
with torch.no_grad():
    for inputs, targets in tqdm(test_loader, desc="Collecting predictions"):
        inputs, targets = inputs.to(device), targets.to(device)
        outputs = model(inputs)
        _, preds = outputs.max(1)
        
        all_preds.extend(preds.cpu().numpy())
        all_targets.extend(targets.cpu().numpy())

# 打印分类报告
print("\n分类报告:")
print(classification_report(all_targets, all_preds))

# 绘制混淆矩阵
plt.figure(figsize=(10, 8))
cm = confusion_matrix(all_targets, all_preds)
sns.heatmap(cm, annot=False, fmt='d', cmap='Blues')
plt.xlabel('预测标签')
plt.ylabel('真实标签')
plt.title('混淆矩阵')
plt.show()

In [ ]:
# 第八个单元格：保存模型推理函数，修复类别数问题
def predict_with_model(model_path, input_data, num_classes=102, scaler_path=None, device='cuda'):
    """
    使用保存的模型进行预测
    
    参数:
        model_path: 模型权重文件路径
        input_data: 输入数据 [n_samples, n_features]，未标准化
        num_classes: 模型输出类别数，默认为102 (0-101)
        scaler_path: scaler路径，如果需要标准化数据
        device: 使用的设备，'cuda'或'cpu'
        
    返回:
        预测的类别概率 [n_samples, n_classes]
    """
    # 如果提供了scaler路径，应用标准化
    if scaler_path is not None:
        from data_loader import load_scaler
        try:
            scaler = load_scaler(scaler_path)
            print(f"使用scaler: {scaler_path}")
            input_data = scaler.transform(input_data)
        except Exception as e:
            print(f"加载scaler失败: {e}")
            print("继续使用未标准化的数据...")
    
    # 转换为torch张量
    if isinstance(input_data, np.ndarray):
        input_data = torch.FloatTensor(input_data)
    
    # 加载模型
    feature_dim = input_data.shape[1]
    
    # 如果未提供num_classes，尝试从模型状态推断
    if num_classes is None:
        # 先加载模型状态字典
        state_dict = torch.load(model_path, map_location=device)
        # 查找输出层权重的形状
        for key in state_dict.keys():
            if 'output_layer.weight' in key:
                num_classes = state_dict[key].shape[0]
                print(f"从模型状态推断类别数: {num_classes}")
                break
        # 如果未找到，使用默认值
        if num_classes is None:
            print("警告: 无法从模型状态推断类别数，使用默认值102")
            num_classes = 102
    
    model = DenseModel(input_dim=feature_dim, hidden_dim=4096, num_classes=num_classes)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.to(device)
    model.eval()
    
    # 预测
    with torch.no_grad():
        input_tensor = input_data.to(device)
        predictions = model(input_tensor)
    
    return predictions.cpu().numpy()

# 示例：如何使用上面的函数进行预测
"""
# 1. 在创建新scaler的情况下 - 需要将数据和scaler一起保存
# 先创建和训练模型
train_loader, val_loader, test_loader, feature_dim, num_classes, scaler = load_and_prepare_data(
    base_dir='/path/to/data',
    create_new_scaler=True  # 创建新的scaler
)
# scaler会被保存到 /path/to/data/scalers/ 目录下

# 2. 稍后使用模型和scaler进行预测
import numpy as np
# 加载新的未标准化数据
new_data = np.random.randn(100, 341)  # 示例数据

# 使用模型进行预测，自动应用标准化
predictions = predict_with_model(
    model_path='dense_4x4096_model_pytorch.pth',
    input_data=new_data,
    num_classes=102,  # 确保与训练时使用的类别数一致 (0-101)
    scaler_path='/path/to/data/scalers/brain_voxel_scaler_latest.pkl'  # 使用latest版本
)

# 获取预测的类别
predicted_classes = np.argmax(predictions, axis=1)
"""

In [ ]:
# 第九个单元格：可视化模型的特征重要性
def compute_feature_importance(model, test_loader, device, num_features=341):
    """
    计算模型的特征重要性，通过对输入特征进行扰动并观察输出变化
    
    这是一个简单的替代方法，用于替代原来的Keras-vis saliency可视化
    """
    model.eval()
    feature_importance = np.zeros(num_features)
    
    # 收集一定数量的样本
    inputs_list = []
    preds_list = []
    
    with torch.no_grad():
        for inputs, _ in tqdm(test_loader, desc="收集样本"):
            inputs = inputs.to(device)
            outputs = model(inputs)
            _, preds = outputs.max(1)
            
            inputs_list.append(inputs.cpu().numpy())
            preds_list.append(preds.cpu().numpy())
            
            # 限制样本数量
            if len(inputs_list) >= 10:  # 只用10个批次的数据
                break
    
    inputs = np.vstack(inputs_list)
    preds = np.concatenate(preds_list)
    
    # 选择一些用于分析的样本索引
    indices_to_visualize = [10, 100, 1000, 2000, 3000]
    
    for idx in indices_to_visualize:
        if idx >= len(inputs):
            continue
            
        input_sample = torch.FloatTensor(inputs[idx:idx+1]).to(device)
        pred_class = preds[idx]
        
        print(f"分析样本 #{idx}, 预测类别: {pred_class}")
        
        # 获取原始预测
        with torch.no_grad():
            original_output = model(input_sample)
            original_prob = original_output[0, pred_class].item()
        
        # 对每个特征进行扰动
        importance = np.zeros(num_features)
        for feat_idx in tqdm(range(num_features), desc=f"扰动特征 (样本 #{idx})"):
            # 创建扰动输入
            perturbed_input = input_sample.clone()
            perturbed_input[0, feat_idx] = 0  # 将特征值设为0
            
            # 获取扰动后的预测
            with torch.no_grad():
                perturbed_output = model(perturbed_input)
                perturbed_prob = perturbed_output[0, pred_class].item()
            
            # 计算影响
            importance[feat_idx] = original_prob - perturbed_prob
        
        # 可视化
        plt.figure(figsize=(12, 6))
        plt.title(f"样本 #{idx}, 预测类别: {pred_class}")
        plt.bar(range(num_features), importance)
        plt.axvspan(0, 15, color='gray', alpha=0.3)
        plt.axvspan(225, 230, color='gray', alpha=0.2)
        plt.xlabel('特征索引')
        plt.ylabel('重要性')
        plt.show()
        
        # 累加到全局特征重要性
        feature_importance += np.abs(importance)
    
    # 归一化全局特征重要性
    feature_importance = feature_importance / len(indices_to_visualize)
    
    # 可视化全局特征重要性
    plt.figure(figsize=(12, 6))
    plt.title("全局特征重要性")
    plt.bar(range(num_features), feature_importance)
    plt.axvspan(0, 15, color='gray', alpha=0.3)
    plt.axvspan(225, 230, color='gray', alpha=0.2)
    plt.xlabel('特征索引')
    plt.ylabel('平均重要性')
    plt.show()
    
    return feature_importance

# 在训练完成后可以运行此函数来分析特征重要性
# feature_importance = compute_feature_importance(model, test_loader, device)